In [8]:
import json
import re
import glob
import os

FILES = glob.glob("./Knowledge-graph/doc_146120936/*-Output*.json")

def extract_triplet_from_raw(raw_output):
    """Extract the first triplet from raw_output text."""
    if not raw_output:
        return None
    
    # Try multiple patterns to extract triplet
    patterns = [
        r'\[subject:[^,\]]+,\s*[^,\]]+,\s*object:[^\]]+\]',  # Full format
        r'\[subject:[^\]]+\]',  # Just subject format
        r'\[[^\]]*subject[^\]]*\]'  # Any format with "subject"
    ]
    
    for pattern in patterns:
        match = re.search(pattern, raw_output, re.IGNORECASE)
        if match:
            return match.group()
    
    return None

for file in FILES:
    print(f"Processing: {file}")
    
    try:
        with open(file, 'r', encoding="utf-8") as f:
            data = json.load(f)
    except Exception as e:
        print(f"  Error loading {file}: {e}")
        continue
    
    # Check if data has the expected structure
    if "results" not in data:
        print(f"  Warning: 'results' key not found in {file}")
        continue
    
    processed_count = 0
    for item in data["results"]:
        if "triplets" in item and isinstance(item["triplets"], dict):
            raw_output = item["triplets"].get("raw_output", "")
            
            if raw_output:
                # Extract triplet using our function
                extracted_triplet = extract_triplet_from_raw(raw_output)
                
                if extracted_triplet:
                    # Ensure triplets is a list
                    if not isinstance(item["triplets"].get("triplets"), list):
                        item["triplets"]["triplets"] = []
                    
                    # Add extracted triplet if not already there
                    if extracted_triplet not in item["triplets"]["triplets"]:
                        item["triplets"]["triplets"].append(extracted_triplet)
                    
                    # REMOVE raw_output only when triplet is extracted (ADDED)
                    if "raw_output" in item["triplets"]:
                        del item["triplets"]["raw_output"]
                    
                    processed_count += 1
    
    # Save modified data
    output_file = f'clean_{os.path.basename(file)}'
    
    try:
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(data, f, indent=2, ensure_ascii=False)
        
        print(f"  Saved to: {output_file}")
        print(f"  Extracted triplets from {processed_count} items")
    except Exception as e:
        print(f"  Error saving {output_file}: {e}")

print("\nAll files processed!")

Processing: ./Knowledge-graph/doc_146120936\LoRA-Output-4B_doc_146120936.json
  Saved to: clean_LoRA-Output-4B_doc_146120936.json
  Extracted triplets from 43 items
Processing: ./Knowledge-graph/doc_146120936\RAG-Output-4B_doc_146120936.json
  Saved to: clean_RAG-Output-4B_doc_146120936.json
  Extracted triplets from 6 items
Processing: ./Knowledge-graph/doc_146120936\RAG-Output_doc_146120936.json
  Saved to: clean_RAG-Output_doc_146120936.json
  Extracted triplets from 1 items

All files processed!
